In [11]:
# Install correct stable versions
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
  Using cached langchain-0.1.17-py3-none-any.whl.metadata (13 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.6/867.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Upload multiple PDFs
from google.colab import files
uploaded = files.upload()

Saving Artificial_intelligence_in_India.pdf to Artificial_intelligence_in_India.pdf
Saving Climate_change_in_India.pdf to Climate_change_in_India.pdf
Saving COVID-19_pandemic.pdf to COVID-19_pandemic.pdf
Saving ISRO.pdf to ISRO.pdf
Saving Temples_of_modern_India.pdf to Temples_of_modern_India.pdf


In [2]:
# Create data folder and move PDFs into it
import os, shutil

os.makedirs("data", exist_ok=True)

for file in uploaded:
    shutil.move(file, "data/" + file)

print("✅ PDFs uploaded:", os.listdir("data"))

✅ PDFs uploaded: ['Artificial_intelligence_in_India.pdf', 'Climate_change_in_India.pdf', 'Temples_of_modern_India.pdf', 'ISRO.pdf', 'COVID-19_pandemic.pdf']


In [3]:
# Load all PDF documents
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

✅ Documents loaded: 232


In [4]:
# Split documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # bigger chunks = faster
    chunk_overlap=100  # keeps context
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

✅ Total chunks: 974


In [5]:
# Convert text into vector embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings ready


In [6]:
# Store embeddings in FAISS vector DB
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

# Save locally
db.save_local("faiss_index")

print("✅ Database created and saved")

✅ Database created and saved


In [7]:
# Create retrieval + generation pipeline
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Retriever (search top 3 chunks)
retriever = db.as_retriever(search_kwargs={"k": 3})

# Lightweight free model
pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

# Combine retrieval + LLM
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ Chatbot ready")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Chatbot ready


In [11]:
while True:
    query = input("Ask (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Stopped!")
        break

    result = qa(query)

    print("\nAnswer:", result["result"])

    print("\nSources:")
    for doc in result["source_documents"]:
        print(doc.metadata)

Ask (type 'exit' to stop): what is artificial intelligence?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

NITI Aayog Problem to Solution Incubation Test Bed will be established. Microsoft will also help
develop AI-assisted models for diabetic retinopathy screening.[49][50]
NITI Aayog drafted a proposal in 2019 to establish an institutional framework for AI. A cabinet note has
been issued proposing to allocate ₹ 7,500 crore initially over three years for the establishment of 20 AI
adaption centers, five core research institutes, and a cloud computing platform named AIRAWAT.[51][52]
INDIAai, a collaborative initiative of the National E-Government Division and NASSCOM for AI-related
advancements, was introduced by Ravi Shankar Prasad on 30 May 2020. In partnership with Intel and the
Ministry of Education, the Responsible AI for Youth Program was also introduced to foster the
development of AI-related skills.[53] The Yardi 

In [ ]:
x